In [ ]:
from pathlib import Path

import pickle
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from config import (
    TAGS_EXCEL_PATH, DATA_CSV_PART1, DATA_CSV_PART2, TARGET_COL,
    FINAL_WEIGHTS, load_tag_lists,
)

## Данные

- `TAG_JOIN_IND` — техническая колонка. Её удаляем.
- Пропуски в TAG считаем нулями.
- Договоры без заполненных TAG не участвуют в расчёте.
- Ниже выводим число исключённых договоров.

In [ ]:
tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
tag_lists = load_tag_lists(tags_descriptions)

part1 = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
part2 = pd.read_csv(DATA_CSV_PART2, encoding='cp1251', delimiter=',')
data = pd.merge(part1, part2, on='POLICY_ZV', how='inner')
data[TARGET_COL] = data['CLAIMS_PART_DAM_COUNT'].astype(bool).astype(int)
data.set_index('POLICY_ZV', inplace=True)

if 'TAG_JOIN_IND' in data.columns:
    data.drop(columns=['TAG_JOIN_IND'], inplace=True)

data['SUM'] = data.filter(like='TAG_').fillna(0).sum(axis=1)
rows_without_tags = int(data['SUM'].le(0).sum())
data = data[data['SUM'] > 0].copy()

print('Исключено договоров без TAG:', rows_without_tags)
print('Осталось строк:', len(data))

## Сырой скор

Для `auto_lover` и `shopping` умножаем каждый TAG на его вес и складываем. Для `alcohol` просто складываем значения TAG.

In [ ]:
raw_scores = pd.DataFrame(index=data.index)
feature_tags = {}
weights = {}

group_settings = {
    'auto_lover': 'auto_lover_list',
    'shopping': 'shopping_features_list',
}

for group_name, tag_list_key in group_settings.items():
    selected_tags = list(tag_lists[tag_list_key])
    group_weights = {
        tag: float(FINAL_WEIGHTS[group_name].get(tag, 0.0))
        for tag in selected_tags
    }
    weights_series = pd.Series(group_weights, dtype=float)
    X = data.reindex(columns=selected_tags, fill_value=0).fillna(0).astype(float)
    raw_scores[f'{group_name}_raw_score'] = (
        X.mul(weights_series, axis=1).sum(axis=1)
    )
    feature_tags[group_name] = selected_tags
    weights[group_name] = group_weights

alcohol_tags = list(tag_lists['alcohol_features_list'])
X_alcohol = data.reindex(columns=alcohol_tags, fill_value=0).fillna(0).astype(float)
raw_scores['alcohol_raw_score'] = X_alcohol.sum(axis=1)
feature_tags['alcohol'] = alcohol_tags

raw_scores.head()

## MinMaxScaler

Для каждой группы создаём отдельный MinMaxScaler. Здесь он настраивается на исходном датасете через `fit_transform`. Параметр `clip=True` ограничит будущие значения диапазоном от 0 до 1.

In [ ]:
result_fit = pd.DataFrame(index=data.index)
scalers = {}

for group_name in ('auto_lover', 'shopping', 'alcohol'):
    raw_column = f'{group_name}_raw_score'
    scaler = MinMaxScaler(clip=True)
    result_fit[f'{group_name}_agg_coef'] = scaler.fit_transform(
        raw_scores[[raw_column]]
    ).ravel()
    scalers[group_name] = scaler

result_fit.head()

## Сохранение

Сохраняем списки TAG, веса и обученные MinMaxScaler в `aggregated_tags_pipeline.pkl`. Рассчитанные признаки сохраняем в CSV.

In [ ]:
artifact = {
    'version': '1.0',
    'feature_tags': feature_tags,
    'weights': weights,
    'scalers': scalers,
}

MODEL_DIR = Path('artifacts') / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'aggregated_tags_pipeline.pkl'
with MODEL_PATH.open('wb') as file:
    pickle.dump(artifact, file, protocol=pickle.HIGHEST_PROTOCOL)

CSV_DIR = Path('artifacts') / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = CSV_DIR / 'all_groups_agg_coef.csv'
result_fit.to_csv(CSV_PATH, index=True, encoding='utf-8-sig')

print('Сохранён pkl:', MODEL_PATH)
print('Сохранён CSV:', CSV_PATH)
print('Min/max:')
for name, scaler in scalers.items():
    print(name, 'min =', scaler.data_min_[0], 'max =', scaler.data_max_[0])

## Проверка

In [ ]:
with MODEL_PATH.open('rb') as file:
    saved = pickle.load(file)
result_transform = pd.DataFrame(index=data.index)

for group_name in ('auto_lover', 'shopping'):
    selected_tags = saved['feature_tags'][group_name]
    weights_series = pd.Series(saved['weights'][group_name], dtype=float)
    X = data.reindex(columns=selected_tags, fill_value=0).fillna(0).astype(float)
    raw_score = X.mul(weights_series, axis=1).sum(axis=1)
    scaled = saved['scalers'][group_name].transform(
        raw_score.to_frame(name=f'{group_name}_raw_score')
    ).ravel()
    result_transform[f'{group_name}_agg_coef'] = scaled

selected_tags = saved['feature_tags']['alcohol']
X = data.reindex(columns=selected_tags, fill_value=0).fillna(0).astype(float)
raw_score = X.sum(axis=1)
scaled = saved['scalers']['alcohol'].transform(
    raw_score.to_frame(name='alcohol_raw_score')
).ravel()
result_transform['alcohol_agg_coef'] = scaled

pd.testing.assert_frame_equal(
    result_fit,
    result_transform,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)
assert result_transform.filter(like='_agg_coef').ge(0).all().all()
assert result_transform.filter(like='_agg_coef').le(1).all().all()
print('Проверка пройдена: transform совпадает с fit_transform.')